# Feature engineering — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D004_C3_Feature_Engineering.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Chapter 3: Feature Engineering

> **Data** volunteers.csv  (exercises use clinic_referrals.csv) · **Time** 45 min · **Needs first** C1, C2

## What you will be able to do

1. Build new numeric features from arithmetic on existing columns
2. Extract year, month, dayofweek and quarter from a datetime column
3. Turn free text into features with CountVectorizer and TfidfVectorizer
4. Bin a continuous column with pd.cut and pd.qcut, and say which one fits
5. Generate interaction and polynomial terms, and state the cost in columns

## The one idea to hold on to

A column can hold real signal in a shape no model can use. Feature engineering is changing the shape, not inventing the signal.

Welcome to the Feature Engineering chapter! Feature engineering is often considered the **most important step** in the machine learning pipeline. It involves transforming raw data into features that better represent the underlying problem, leading to improved model accuracy.

---

## What is Feature Engineering?

**Feature Engineering** is the process of using domain knowledge to extract, transform, and create new features (variables) from raw data that make machine learning algorithms work better.

![Feature Engineering Overview](https://miro.medium.com/v2/resize:fit:1400/1*p3jlS0M2AoKNZlyWxfxuNA.png)

*The feature engineering process transforms raw data into meaningful features for ML models.*

### Why is Feature Engineering Important?

| Benefit | Description |
| :--- | :--- |
| **Improved Accuracy** | Well-engineered features can significantly boost model performance |
| **Reduced Overfitting** | Proper features help models generalize better |
| **Faster Training** | Relevant features reduce computational complexity |
| **Better Interpretability** | Meaningful features make models easier to understand |

> "Coming up with features is difficult, time-consuming, and requires expert knowledge. Applied machine learning is basically feature engineering." - Andrew Ng

In [ ]:
# TODO: call `warnings.filterwarnings`

In [ ]:
# ═══ CHECKPOINT — "Loading the Dataset" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import PolynomialFeatures
import warnings

print("Checkpoint restored. Ready for: Loading the Dataset")

## 1. Loading the Dataset

We will work with the `volunteers.csv` dataset, which contains information about volunteer opportunities in New York City.

In [ ]:
# TODO: compute `df`
# Name your result: df

In [3]:
# Check data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 665 entries, 0 to 664
Data columns (total 35 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   opportunity_id      665 non-null    int64  
 1   content_id          665 non-null    int64  
 2   vol_requests        665 non-null    int64  
 3   event_time          665 non-null    int64  
 4   title               665 non-null    object 
 5   hits                665 non-null    int64  
 6   summary             665 non-null    object 
 7   is_priority         62 non-null     object 
 8   category_id         617 non-null    float64
 9   category_desc       617 non-null    object 
 10  amsl                0 non-null      float64
 11  amsl_unit           0 non-null      float64
 12  org_title           665 non-null    object 
 13  org_content_id      665 non-null    int64  
 14  addresses_count     665 non-null    int64  
 15  locality            595 non-null    object 
 16  region  

In [ ]:
# TODO: write the code for this step

---

## 2. Feature Creation from Existing Data

One of the most powerful feature engineering techniques is **creating new features** by combining or transforming existing ones.

![Feature Creation](https://www.researchgate.net/publication/344645696/figure/fig1/AS:945715506655241@1602477254963/Feature-engineering-pipeline.png)

*Creating new features from existing data can reveal hidden patterns.*

### 2.1 Mathematical Operations

We can create features using arithmetic operations on numerical columns.

In [ ]:
# TODO: compute `df_eng` from `hits_per_request`, `hits`, `vol_requests`
# Name your result: df_eng

In [ ]:
# TODO: compute `median_hits` from `hits`, `high_engagement`
# Name your result: median_hits

In [ ]:
# ═══ CHECKPOINT — "String/Text Manipulation" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import PolynomialFeatures
import warnings
df = pd.read_csv('../../00-data/volunteers.csv')
df_eng = df.copy()
df_eng['hits_per_request'] = df_eng['hits'] / (df_eng['vol_requests'] + 1)
median_hits = df_eng['hits'].median()
df_eng['high_engagement'] = (df_eng['hits'] > median_hits).astype(int)

print("Checkpoint restored. Ready for: String/Text Manipulation")

### 2.2 String/Text Manipulation

We can extract useful information from text columns.

In [7]:
# Create feature: length of the title
df_eng['title_length'] = df_eng['title'].str.len()

# Create feature: word count in title
df_eng['title_word_count'] = df_eng['title'].str.split().str.len()

# Create feature: length of summary
df_eng['summary_length'] = df_eng['summary'].fillna('').str.len()

print("Text-based features created!")
df_eng[['title', 'title_length', 'title_word_count', 'summary_length']].head()

Text-based features created!


,title,title_length,title_word_count,summary_length
0,Volunteers Needed For Rise Up & Stay Put! Home...,59,11,291
1,Web designer,12,2,38
2,Urban Adventures - Ice Skating at Lasker Rink,45,8,114
3,Fight global hunger and support women farmers ...,83,15,281
4,Stop 'N' Swap,13,3,171


---

## 3. DateTime Feature Engineering

Datetime columns contain rich information that can be extracted into multiple meaningful features.

![DateTime Features](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*X7NYLmVlLNp9zGCdTPNwsQ.png)

*Extracting temporal features from datetime columns.*

### Common DateTime Features:
- Year, Month, Day
- Day of Week
- Hour, Minute
- Is Weekend
- Quarter
- Time since a reference date

In [8]:
# Check date columns
date_cols = ['created_date', 'start_date_date', 'end_date_date', 'last_modified_date']
print("Sample date values:")
df_eng[date_cols].head()

Sample date values:


,created_date,start_date_date,end_date_date,last_modified_date
0,January 13 2011,July 30 2011,July 30 2011,June 23 2011
1,January 14 2011,February 01 2011,February 01 2011,January 25 2011
2,January 19 2011,January 29 2011,January 29 2011,January 21 2011
3,January 21 2011,February 14 2011,March 31 2012,January 25 2011
4,January 28 2011,February 05 2011,February 05 2011,February 01 2011


In [ ]:
# TODO: call `pd.to_datetime`

In [10]:
# Extract datetime features from 'start_date_date'
df_eng['start_year'] = df_eng['start_date_date'].dt.year
df_eng['start_month'] = df_eng['start_date_date'].dt.month
df_eng['start_day'] = df_eng['start_date_date'].dt.day
df_eng['start_dayofweek'] = df_eng['start_date_date'].dt.dayofweek  # 0=Monday, 6=Sunday
df_eng['start_is_weekend'] = (df_eng['start_dayofweek'] >= 5).astype(int)
df_eng['start_quarter'] = df_eng['start_date_date'].dt.quarter

print("DateTime features extracted!")
df_eng[['start_date_date', 'start_year', 'start_month', 'start_day', 
        'start_dayofweek', 'start_is_weekend', 'start_quarter']].head(10)

DateTime features extracted!


,start_date_date,start_year,start_month,start_day,start_dayofweek,start_is_weekend,start_quarter
0,2011-07-30,2011,7,30,5,1,3
1,2011-02-01,2011,2,1,1,0,1
2,2011-01-29,2011,1,29,5,1,1
3,2011-02-14,2011,2,14,0,0,1
4,2011-02-05,2011,2,5,5,1,1
5,2011-02-12,2011,2,12,5,1,1
6,2011-01-27,2011,1,27,3,0,1
7,2011-02-07,2011,2,7,0,0,1
8,2011-02-08,2011,2,8,1,0,1
9,2011-02-14,2011,2,14,0,0,1


In [11]:
# Create feature: duration of the opportunity (in days)
df_eng['duration_days'] = (df_eng['end_date_date'] - df_eng['start_date_date']).dt.days

# Create feature: days from creation to start
df_eng['days_to_start'] = (df_eng['start_date_date'] - df_eng['created_date']).dt.days

print("Duration features created!")
df_eng[['created_date', 'start_date_date', 'end_date_date', 'duration_days', 'days_to_start']].head(10)

Duration features created!


,created_date,start_date_date,end_date_date,duration_days,days_to_start
0,2011-01-13,2011-07-30,2011-07-30,0,198
1,2011-01-14,2011-02-01,2011-02-01,0,18
2,2011-01-19,2011-01-29,2011-01-29,0,10
3,2011-01-21,2011-02-14,2012-03-31,411,24
4,2011-01-28,2011-02-05,2011-02-05,0,8
5,2011-01-28,2011-02-12,2011-02-12,0,15
6,2011-01-27,2011-01-27,2012-01-27,365,0
7,2011-02-07,2011-02-07,2012-02-07,365,0
8,2011-02-09,2011-02-08,2012-02-08,365,-1
9,2011-02-09,2011-02-14,2012-02-09,360,5


In [ ]:
# ═══ CHECKPOINT — "Visualizing DateTime Patterns" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import PolynomialFeatures
import warnings
df = pd.read_csv('../../00-data/volunteers.csv')
df_eng = df.copy()
df_eng['hits_per_request'] = df_eng['hits'] / (df_eng['vol_requests'] + 1)
median_hits = df_eng['hits'].median()
df_eng['high_engagement'] = (df_eng['hits'] > median_hits).astype(int)
df_eng['title_length'] = df_eng['title'].str.len()
df_eng['title_word_count'] = df_eng['title'].str.split().str.len()
df_eng['summary_length'] = df_eng['summary'].fillna('').str.len()
date_cols = ['created_date', 'start_date_date', 'end_date_date', 'last_modified_date']
for col in date_cols:
    df_eng[col] = pd.to_datetime(df_eng[col], format='%B %d %Y', errors='coerce')
df_eng['start_year'] = df_eng['start_date_date'].dt.year
df_eng['start_month'] = df_eng['start_date_date'].dt.month
df_eng['start_day'] = df_eng['start_date_date'].dt.day
df_eng['start_dayofweek'] = df_eng['start_date_date'].dt.dayofweek
df_eng['start_is_weekend'] = (df_eng['start_dayofweek'] >= 5).astype(int)
df_eng['start_quarter'] = df_eng['start_date_date'].dt.quarter
df_eng['duration_days'] = (df_eng['end_date_date'] - df_eng['start_date_date']).dt.days
df_eng['days_to_start'] = (df_eng['start_date_date'] - df_eng['created_date']).dt.days

print("Checkpoint restored. Ready for: Visualizing DateTime Patterns")

### Visualizing DateTime Patterns

In [ ]:
# TODO: draw a `plt.subplots` of `start_dayofweek`
# Name your result: day_names, day_counts, month_counts

---

## 4. Text Feature Engineering

Text data requires special techniques to convert into numerical features that ML models can use.

### 4.1 Bag of Words (BoW)

The Bag of Words model represents text as a vector of word counts, ignoring grammar and word order.

![Bag of Words](https://miro.medium.com/v2/resize:fit:1400/1*cTt3HXIv2NZS5GnXP94fTg.png)

*Bag of Words converts text into numerical vectors based on word frequency.*

In [ ]:
# TODO: compute `sample_texts`, `count_vec`, `bow_matrix`
# Name your result: sample_texts, count_vec, bow_matrix, bow_df

### 4.2 TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF weighs words by their importance - words that appear frequently in one document but rarely across all documents are given higher weight.

**Formula:**
- **TF (Term Frequency)** = (Number of times term t appears in document) / (Total terms in document)
- **IDF (Inverse Document Frequency)** = log(Total documents / Documents containing term t)
- **TF-IDF** = TF x IDF

![TF-IDF](https://miro.medium.com/v2/resize:fit:1400/1*qQgnyPLDIkUmeZKN2_ZWbQ.png)

*TF-IDF gives higher weight to distinctive words.*

In [ ]:
# TODO: compute `tfidf_vec`, `tfidf_matrix`, `tfidf_df`
# Name your result: tfidf_vec, tfidf_matrix, tfidf_df, columns

In [ ]:
# TODO: compute `tfidf_titles`, `title_features`, `title_tfidf_df` from `title`
# Name your result: tfidf_titles, title_features, title_tfidf_df, columns

---

## 5. Binning and Discretization

Binning transforms continuous variables into categorical bins, which can help capture non-linear relationships.

![Binning](https://miro.medium.com/v2/resize:fit:1200/1*W51t4_1zyoLg1IhTbR1Xdw.png)

*Converting continuous data into discrete bins.*

In [ ]:
# TODO: compute `bins`, `labels` from `hits_category`, `hits`
# Name your result: bins, labels

In [ ]:
# TODO: compute `q`, `labels` from `hits_quantile`, `hits`
# Name your result: q, labels

In [ ]:
# TODO: draw a `plt.subplots` of `hits_category`

---

## 6. Polynomial Features

Polynomial features create interactions and higher-order terms to capture non-linear relationships.

In [ ]:
# TODO: compute `poly`, `sample_data`, `poly_features`
# Name your result: poly, sample_data, poly_features, poly_df

---

## 7. Summary of Engineered Features

Let us review all the features we have created:

In [ ]:
# TODO: write the code for this step
# Name your result: original_cols, new_cols

---

## Exercises

### Exercise 1: Create New Features
Create a feature called `title_has_urgent` that is 1 if the title contains the word "urgent" (case-insensitive) and 0 otherwise.

In [21]:
# Your code here
# df_eng['title_has_urgent'] = ...

In [ ]:
# ═══ CHECKPOINT — "Exercise 2: DateTime Features" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import PolynomialFeatures
import warnings
df = pd.read_csv('../../00-data/volunteers.csv')
df_eng = df.copy()
df_eng['hits_per_request'] = df_eng['hits'] / (df_eng['vol_requests'] + 1)
median_hits = df_eng['hits'].median()
df_eng['high_engagement'] = (df_eng['hits'] > median_hits).astype(int)
df_eng['title_length'] = df_eng['title'].str.len()
df_eng['title_word_count'] = df_eng['title'].str.split().str.len()
df_eng['summary_length'] = df_eng['summary'].fillna('').str.len()
date_cols = ['created_date', 'start_date_date', 'end_date_date', 'last_modified_date']
for col in date_cols:
    df_eng[col] = pd.to_datetime(df_eng[col], format='%B %d %Y', errors='coerce')
df_eng['start_year'] = df_eng['start_date_date'].dt.year
df_eng['start_month'] = df_eng['start_date_date'].dt.month
df_eng['start_day'] = df_eng['start_date_date'].dt.day
df_eng['start_dayofweek'] = df_eng['start_date_date'].dt.dayofweek
df_eng['start_is_weekend'] = (df_eng['start_dayofweek'] >= 5).astype(int)
df_eng['start_quarter'] = df_eng['start_date_date'].dt.quarter
df_eng['duration_days'] = (df_eng['end_date_date'] - df_eng['start_date_date']).dt.days
df_eng['days_to_start'] = (df_eng['start_date_date'] - df_eng['created_date']).dt.days
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = df_eng['start_dayofweek'].value_counts().sort_index()
month_counts = df_eng['start_month'].value_counts().sort_index()
sample_texts = ['Volunteers needed for community garden project', 'Help needed at food bank distribution center', 'Community volunteers wanted for park cleanup', 'Food distribution volunteers needed urgently']
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(sample_texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vec.get_feature_names_out(), index=[f'Doc {i + 1}' for i in range(len(sample_texts))])
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(sample_texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray().round(3), columns=tfidf_vec.get_feature_names_out(), index=[f'Doc {i + 1}' for i in range(len(sample_texts))])
tfidf_titles = TfidfVectorizer(max_features=10, stop_words='english')
title_features = tfidf_titles.fit_transform(df_eng['title'].fillna(''))
title_tfidf_df = pd.DataFrame(title_features.toarray(), columns=[f'tfidf_{word}' for word in tfidf_titles.get_feature_names_out()])
df_eng['hits_category'] = pd.cut(df_eng['hits'], bins=[0, 100, 500, 1000, float('inf')], labels=['Low', 'Medium', 'High', 'Very High'])
df_eng['hits_quantile'] = pd.qcut(df_eng['hits'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
poly = PolynomialFeatures(degree=2, include_bias=False)
sample_data = df_eng[['hits', 'vol_requests']].fillna(0).head(5)
poly_features = poly.fit_transform(sample_data)
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(['hits', 'vol_requests']))
original_cols = set(df.columns)
new_cols = [col for col in df_eng.columns if col not in original_cols]

print("Checkpoint restored. Ready for: Exercise 2: DateTime Features")

### Exercise 2: DateTime Features
Extract the day name (e.g., 'Monday', 'Tuesday') from the `start_date_date` column.

In [22]:
# Your code here
# df_eng['start_day_name'] = ...

### Exercise 3: Binning
Create bins for `vol_requests` with categories: 'Few' (1-10), 'Moderate' (11-50), 'Many' (51-100), 'Large Scale' (100+).

In [23]:
# Your code here
# df_eng['volunteer_scale'] = ...

### Exercise 4: Text Features
Apply Bag of Words to the `summary` column and extract the top 5 most common words.

In [24]:
# Your code here

---

## Key Takeaways

1. **Feature Engineering is crucial** - It can make or break your ML model's performance
2. **Domain knowledge helps** - Understanding your data leads to better features
3. **DateTime features are rich** - Extract year, month, day, weekday, duration, etc.
4. **Text requires conversion** - Use BoW or TF-IDF to convert text to numbers
5. **Binning captures non-linearity** - Discretization can help certain models
6. **Polynomial features** - Capture interactions between variables

### Related Chapters

For more specialized preprocessing techniques, see:
- **Chapter 2: Feature Scaling** - StandardScaler, MinMaxScaler, RobustScaler
- **Chapter 4: Categorical Encoding** - One-Hot, Label, Ordinal, Frequency encoding

---

## Additional Resources

- [Scikit-learn Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Feature Engineering for Machine Learning by Alice Zheng](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/)
- [Kaggle Feature Engineering Course](https://www.kaggle.com/learn/feature-engineering)

---

*End of Chapter 3: Feature Engineering*

---

## How this is used in real work

This is where most real accuracy comes from, and it is the part that does not transfer between problems. `dayofweek` on a transaction, `days_since_last_seen` on a user, TF-IDF on a support ticket: the library calls are trivial and knowing which one to reach for is the job.

## Where this goes next

**C3b** for anything with a time axis. **C4** encodes what is still text.